In [1]:
import openeo
from openeo.processes import quantiles
import leafmap
from shapely.geometry import shape
from folium.plugins import Draw
from IPython.display import JSON
import urllib
import json
import numpy as np
import pandas as pd

In [3]:
# define properties for the outputs
from pathlib import Path

out_dir = Path("/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/glacier/suldenferner")
mkdir = out_dir.mkdir(parents=True, exist_ok=True)

event = "Suldenferner_2026"

BANDS = ["B02", "B03", "B04", "B08", "B12" ,"SCL"]  

In [4]:
PRE_DATE  = ("2026-07-01", "2026-07-25")   
POST_DATE = ("2026-07-27", "2026-07-30")   

In [5]:
# open a map and zoom to the area of interest
m = leafmap.Map(center=(46.65, 11.4), zoom=8.5)
m

Map(center=[46.65, 11.4], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

In [6]:
feat = m.draw_features
geom_dict = feat[0]['geometry']
geom = shape(geom_dict)

minx, miny, maxx, maxy = geom.bounds

bbox = {
    "west": minx,
    "south": miny,
    "east": maxx,
    "north": maxy,
}

print(bbox)

{'west': 10.558333, 'south': 46.484742, 'east': 10.584211, 'north': 46.499249}


In [7]:
# load sentinel-2 data after the flood event
connection = openeo.connect("openeo.dataspace.copernicus.eu").authenticate_oidc()
connection.authenticate_oidc()

# Load Sentinel-2 data
def load_s2(temporal_extent):
    cube = connection.load_collection(
        "SENTINEL2_L2A",
        spatial_extent=bbox,
        temporal_extent=list(temporal_extent),
        bands=BANDS,
        max_cloud_cover=50,
    )
    return cube

pre_cube=load_s2(PRE_DATE)
post_cube = load_s2(POST_DATE)

Authenticated using refresh token.
Authenticated using refresh token.


In [8]:
def mask_clouds(cube):
    scl = cube.band("SCL")
    return (
    (scl == 1) |
    (scl == 3) |
    (scl == 7) |
    (scl == 8) |
    (scl == 9) |
    (scl == 10)
    )

In [9]:
post_cube_mask = mask_clouds(post_cube)
post_cube_masked = post_cube.mask(post_cube_mask).mean_time()

In [10]:
file_post_cube = out_dir/ f"post_{event}.tif"

print(file_post_cube)

post_cube_masked.download(file_post_cube, format='GTiff')

/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/glacier/suldenferner/post_Suldenferner_2026.tif


In [11]:
pre_cube_mask = mask_clouds(pre_cube)

pre_cube_masked = pre_cube.mask(pre_cube_mask).reduce_dimension(
    dimension='t',
    reducer=lambda x: quantiles(data=x, probabilities=[0.25])
)

In [12]:
file_pre_cube = out_dir/ f"pre_mosaic_{event}.tif"

print(file_pre_cube)

pre_cube_masked.download(file_pre_cube, format='GTiff')

/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/glacier/suldenferner/pre_mosaic_Suldenferner_2026.tif


In [14]:
# calcualte NDSI for post-flood period
def compute_ndsi(cube):
    swir = cube.band("B12")
    green = cube.band("B03")
    return (swir - green) / (swir + green)

post_ndsi = compute_ndsi(post_cube_masked)
pre_ndsi = compute_ndsi(pre_cube_masked)

In [15]:
file_pre_ndsi = out_dir/ f"pre_mosaic_{event}_NDSI.tif"
file_post_ndsi = out_dir/ f"post_{event}_NDSI.tif"

pre_ndsi.download(file_pre_ndsi)
post_ndsi.download(file_post_ndsi)